# Tennis Serve Phase Classifier — Google Colab

Запускать на T4 GPU (Runtime → Change runtime type → T4 GPU)

**Порядок:**
1. Загрузить архив с позами и разметкой (из Google Drive)
2. Обучить BiLSTM
3. Сконвертировать в TFLite
4. Скачать `serve_phase.tflite`

In [ ]:
# Установка зависимостей
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q tensorflow==2.16.1 onnx onnx-tf scikit-learn pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

# Укажите путь к архиву в вашем Google Drive
DRIVE_ZIP = '/content/drive/MyDrive/tennis_dataset.zip'

!unzip -q {DRIVE_ZIP} -d /content/dataset
print('Файлы:')
!ls /content/dataset/

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEQUENCE_LEN = 60
STRIDE = 10
INPUT_SIZE = 33 * 4   # x, y, z, visibility
HIDDEN_SIZE = 128
NUM_LAYERS = 2
NUM_CLASSES = 6

PHASE_TO_IDX = {
    'IDLE': 0, 'READY_STANCE': 1, 'TOSS': 2,
    'TROPHY': 3, 'ACCELERATION': 4, 'FOLLOW_THROUGH': 5
}
IDX_TO_PHASE = {v: k for k, v in PHASE_TO_IDX.items()}

def load_dataset(poses_dir, labels_dir):
    sequences, labels = [], []
    pose_files = list(Path(poses_dir).glob('*.csv'))
    print(f'Видео файлов: {len(pose_files)}')

    for pf in pose_files:
        lf = Path(labels_dir) / f'{pf.stem}_labels.csv'
        if not lf.exists():
            continue
        poses = pd.read_csv(pf)
        lbls = pd.read_csv(lf)
        merged = poses.merge(lbls, on='frame_idx', how='inner')
        if len(merged) < SEQUENCE_LEN:
            continue

        feat_cols = [c for c in merged.columns
                     if any(c.endswith(s) for s in ('_x','_y','_z','_vis'))]
        X = merged[feat_cols].fillna(0).values.astype(np.float32)
        y = merged['phase'].map(PHASE_TO_IDX).fillna(0).values.astype(int)

        for start in range(0, len(X) - SEQUENCE_LEN, STRIDE):
            sequences.append(X[start:start+SEQUENCE_LEN])
            labels.append(int(np.bincount(y[start:start+SEQUENCE_LEN]).argmax()))

    X_arr, y_arr = np.array(sequences), np.array(labels)
    print(f'Последовательностей: {len(X_arr)}')
    print(f'Распределение классов: {np.bincount(y_arr)}')
    return X_arr, y_arr

X, y = load_dataset('/content/dataset/poses', '/content/dataset/raw_videos')
print(f'Shape: {X.shape}')

In [ ]:
class ServeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class ServeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS,
                            batch_first=True, dropout=0.3, bidirectional=True)
        self.head = nn.Sequential(
            nn.Linear(HIDDEN_SIZE * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
train_dl = DataLoader(ServeDataset(X_tr, y_tr), batch_size=64, shuffle=True)
val_dl   = DataLoader(ServeDataset(X_val, y_val), batch_size=128)

model = ServeLSTM().to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=50)
loss_fn = nn.CrossEntropyLoss()

print(f'Параметров: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
EPOCHS = 50
best_acc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        opt.step()
        total_loss += loss.item()

    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device), yb.to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)

    acc = correct / total
    sched.step()

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), '/content/serve_phase_best.pt')

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:03d} | loss={total_loss/len(train_dl):.4f} | val_acc={acc:.3f} | best={best_acc:.3f}')

print(f'\nЛучшая точность: {best_acc:.3f}')

In [ ]:
# Матрица ошибок
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

model.load_state_dict(torch.load('/content/serve_phase_best.pt'))
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in val_dl:
        all_preds.extend(model(xb.to(device)).argmax(1).cpu().numpy())
        all_true.extend(yb.numpy())

print(classification_report(all_true, all_preds,
      target_names=list(PHASE_TO_IDX.keys())))

cm = confusion_matrix(all_true, all_preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=PHASE_TO_IDX.keys(),
            yticklabels=PHASE_TO_IDX.keys(), cmap='Blues')
plt.title('Confusion Matrix — Serve Phases')
plt.show()

In [ ]:
# Экспорт PyTorch → ONNX
model.eval()
dummy = torch.zeros(1, SEQUENCE_LEN, INPUT_SIZE)
torch.onnx.export(
    model, dummy, '/content/serve_phase.onnx',
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}},
    opset_version=17
)
print('ONNX экспортирован')

In [ ]:
# ONNX → TensorFlow SavedModel
!pip install -q onnx-tf
import onnx
from onnx_tf.backend import prepare

onnx_model = onnx.load('/content/serve_phase.onnx')
tf_rep = prepare(onnx_model)
tf_rep.export_graph('/content/serve_phase_tf')
print('TF SavedModel готов')

In [ ]:
# TF SavedModel → TFLite с INT8 квантизацией
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model('/content/serve_phase_tf')
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Репрезентативные данные для калибровки
def representative_dataset():
    for i in range(0, min(200, len(X_val)), 1):
        yield [X_val[i:i+1]]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32  # float32 вход удобнее для MediaPipe pipeline
converter.inference_output_type = tf.float32

tflite_model = converter.convert()
with open('/content/serve_phase.tflite', 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f'TFLite модель: {size_kb:.1f} KB')
print('Скачайте serve_phase.tflite и поместите в app/src/main/assets/')

In [ ]:
# Скачать файл
from google.colab import files
files.download('/content/serve_phase.tflite')